In [ ]:
#!/usr/bin/env python3
"""
gsMap Pipeline
================
GWAS + scRNA-seq -> genetically informed spatial mapping.

Steps:
  0a. GWAS QC and formatting
  0b. scRNA-seq QC and formatting
  1.  Find latent representations
  2.  Latent to gene mapping
  3.  Generate LD scores (22 chromosomes in parallel)
  4.  Spatial LDSC
  5.  Cauchy combination
  6.  Causal gene analysis
  7.  Visualization
"""

import os
import sys
import time
import subprocess
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import stats
from pathlib import Path


# ========================= Configuration =========================

# --- Required paths (fill in before running) ---
GWAS_FILE      = "./data/gwas/your_gwas_summary_stats.txt"     # Path to GWAS summary statistics
TRAIT_NAME     = "demo_trait"     # Short trait name
SCRNA_FILE     = "./data/visium/your_scRNA_data.h5ad"     # Path to scRNA-seq h5ad
GSMAP_RESOURCE = "./data/gsmap_resource/"     # gsMap resource directory
PROJECT_DIR    = "./results/"     # Output project directory

# --- GWAS column mapping ---
GWAS_SEP              = "\t"
GWAS_COL_SNP          = "SNP"
GWAS_COL_EFFECT_ALLELE = "effect_allele"
GWAS_COL_OTHER_ALLELE  = "other_allele"
GWAS_COL_BETA         = "beta"
GWAS_COL_SE           = "se"
GWAS_COL_PVAL         = "pval"
GWAS_COL_N            = "N"
GWAS_COL_Z            = None      # Set if Z column exists, otherwise computed from beta/se

# --- scRNA-seq configuration ---
ANNOTATION_COL     = "supercluster_term"    # Cell type annotation column
REGION_COL         = "ROIGroupCoarse"       # Brain region column (optional)
GENE_NAME_COL      = "feature_name"         # Gene symbol column in var
DATA_LAYER         = None
MAX_CELLS_PER_TYPE = 10000
SPECIES            = "human"     # "human", "mouse", or "macaque"

# --- Compute resources ---
NUM_PARALLEL_CHROMS = 22
NUM_PROCESSES_LDSC  = 32


# ========================= Auto-configured =========================

GSMAP_BIN = os.path.join(os.path.dirname(sys.executable), "gsmap")
if not os.path.exists(GSMAP_BIN):
    GSMAP_BIN = "gsmap"


# ========================= Utilities =========================

def log(msg, level="INFO"):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] {level} | {msg}")


def run_gsmap(args, step_name):
    """Run a gsmap CLI command and return success status."""
    log(f"Starting {step_name}...")
    cmd = [GSMAP_BIN] + args
    process = subprocess.run(cmd, capture_output=True, text=True)
    if process.returncode == 0:
        log(f"{step_name} completed")
        return True
    else:
        log(f"{step_name} failed", "ERROR")
        log(process.stderr[-1000:], "ERROR")
        return False


def _run_chrom_global(args):
    """Run LD score generation for a single chromosome (for parallel dispatch)."""
    gsmap_bin, workdir, sample_name, chrom, resource = args
    cmd = [gsmap_bin, "run_generate_ldscore",
           "--workdir", workdir,
           "--sample_name", sample_name,
           "--chrom", str(chrom),
           "--bfile_root", f"{resource}/LD_Reference_Panel/1000G_EUR_Phase3_plink/1000G.EUR.QC",
           "--keep_snp_root", f"{resource}/LDSC_resource/hapmap3_snps/hm",
           "--gtf_annotation_file", f"{resource}/genome_annotation/gtf/gencode.v46lift37.basic.annotation.gtf",
           "--gene_window_size", "50000"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return chrom, result.returncode


# ========================= Step 0a: GWAS QC =========================

def qc_and_format_gwas():
    """QC and format GWAS summary statistics for gsMap."""
    log("=" * 60)
    log("Step 0a: GWAS QC and formatting")
    log("=" * 60)

    output_path = os.path.join(PROJECT_DIR, f"{TRAIT_NAME}.sumstats.gz")
    if os.path.exists(output_path):
        log(f"Formatted GWAS already exists, skipping: {output_path}")
        return output_path

    log(f"Reading GWAS: {GWAS_FILE}")
    gwas = pd.read_csv(GWAS_FILE, sep=GWAS_SEP)
    log(f"Raw: {gwas.shape[0]} rows, columns: {gwas.columns.tolist()}")

    # Check required columns
    required_cols = {
        "SNP": GWAS_COL_SNP,
        "A1": GWAS_COL_EFFECT_ALLELE,
        "A2": GWAS_COL_OTHER_ALLELE,
        "N": GWAS_COL_N,
    }
    for name, col in required_cols.items():
        if col not in gwas.columns:
            log(f"Missing column: {col} (needed as {name})", "ERROR")
            for c in gwas.columns:
                if name.lower() in c.lower() or col.lower() in c.lower():
                    log(f"  Auto-matched: {c} -> {name}")
                    gwas = gwas.rename(columns={c: col})
                    break

    # Compute Z values
    if GWAS_COL_Z and GWAS_COL_Z in gwas.columns:
        log("Using existing Z column")
        gwas["Z"] = gwas[GWAS_COL_Z]
    elif GWAS_COL_BETA in gwas.columns and GWAS_COL_SE in gwas.columns:
        log("Computing Z from beta/se")
        gwas["Z"] = gwas[GWAS_COL_BETA] / gwas[GWAS_COL_SE]
    else:
        log("Cannot compute Z values", "ERROR")
        return None

    # Filter
    n_before = len(gwas)
    gwas = gwas.dropna(subset=[GWAS_COL_SNP, GWAS_COL_EFFECT_ALLELE,
                                GWAS_COL_OTHER_ALLELE, "Z", GWAS_COL_N])
    gwas = gwas[np.isfinite(gwas["Z"])]
    gwas = gwas[gwas["Z"].abs() < 40]
    gwas = gwas[gwas[GWAS_COL_N] > 0]

    valid_alleles = {"A", "T", "C", "G"}
    gwas = gwas[gwas[GWAS_COL_EFFECT_ALLELE].str.upper().isin(valid_alleles)]
    gwas = gwas[gwas[GWAS_COL_OTHER_ALLELE].str.upper().isin(valid_alleles)]
    gwas = gwas.drop_duplicates(subset=[GWAS_COL_SNP], keep="first")

    n_after = len(gwas)
    log(f"QC: {n_before} -> {n_after} SNPs (removed {n_before - n_after})")

    # Format output
    gwas_fmt = pd.DataFrame({
        "SNP": gwas[GWAS_COL_SNP],
        "A1": gwas[GWAS_COL_EFFECT_ALLELE].str.upper(),
        "A2": gwas[GWAS_COL_OTHER_ALLELE].str.upper(),
        "Z": gwas["Z"],
        "N": gwas[GWAS_COL_N],
    })

    gwas_fmt.to_csv(output_path, sep="\t", index=False, compression="gzip")
    log(f"GWAS formatted: {output_path}")
    log(f"  SNPs: {len(gwas_fmt)}, Z range: [{gwas_fmt['Z'].min():.2f}, {gwas_fmt['Z'].max():.2f}]")
    return output_path


# ========================= Step 0b: scRNA-seq QC =========================

def qc_and_format_scrna():
    """QC and format scRNA-seq data for gsMap."""
    log("=" * 60)
    log("Step 0b: scRNA-seq QC and formatting")
    log("=" * 60)

    output_path = os.path.join(PROJECT_DIR, f"scrna_for_gsmap_{TRAIT_NAME}.h5ad")
    if os.path.exists(output_path):
        log(f"Formatted scRNA already exists, skipping: {output_path}")
        return output_path

    log(f"Reading scRNA-seq: {SCRNA_FILE}")
    adata = sc.read_h5ad(SCRNA_FILE, backed="r")
    log(f"Raw: {adata.n_obs} cells, {adata.n_vars} genes")

    # Check annotation column
    if ANNOTATION_COL not in adata.obs.columns:
        log(f"Annotation column not found: {ANNOTATION_COL}", "ERROR")
        log(f"Available columns: {adata.obs.columns.tolist()}")
        return None

    # Downsample per cell type
    log(f"Downsampling: max {MAX_CELLS_PER_TYPE} cells per {ANNOTATION_COL}")
    np.random.seed(42)
    sample_indices = []
    categories = (adata.obs[ANNOTATION_COL].cat.categories
                  if hasattr(adata.obs[ANNOTATION_COL], "cat")
                  else adata.obs[ANNOTATION_COL].unique())
    for ct in categories:
        idx = np.where(adata.obs[ANNOTATION_COL].values == ct)[0]
        n_sample = min(len(idx), MAX_CELLS_PER_TYPE)
        sampled = np.random.choice(idx, n_sample, replace=False)
        sample_indices.extend(sampled)
        log(f"  {ct}: {len(idx)} -> {n_sample}")

    sample_indices = sorted(sample_indices)
    log(f"After downsampling: {len(sample_indices)} cells")

    log("Loading subset into memory...")
    adata_sub = adata[sample_indices, :].to_memory()

    # Gene name conversion
    is_ensembl = adata_sub.var_names[0].startswith("ENSG") or adata_sub.var_names[0].startswith("ENSMUSG")
    if is_ensembl:
        if GENE_NAME_COL in adata_sub.var.columns:
            log(f"Converting Ensembl IDs to {GENE_NAME_COL}")
            adata_sub.var_names = adata_sub.var[GENE_NAME_COL].values.astype(str)
        elif "Gene" in adata_sub.var.columns:
            log("Converting Ensembl IDs to Gene column")
            adata_sub.var_names = adata_sub.var["Gene"].values.astype(str)

    dup_mask = adata_sub.var_names.duplicated(keep="first")
    if dup_mask.sum() > 0:
        log(f"Removing {dup_mask.sum()} duplicate gene names")
        adata_sub = adata_sub[:, ~dup_mask]

    adata_sub.var_names = adata_sub.var_names.astype(str)
    adata_sub.var.index = adata_sub.var.index.astype(str)

    # Expression matrix check
    test_data = adata_sub.X[:100, :100]
    if hasattr(test_data, "toarray"):
        test_data = test_data.toarray()

    is_integer = np.allclose(test_data, np.round(test_data))
    max_val = test_data.max()

    if is_integer and max_val > 5:
        log("Expression matrix: raw counts")
    elif max_val < 30 and not is_integer:
        log("Expression matrix appears log-normalized, applying expm1", "WARN")
        import scipy.sparse as sp
        if sp.issparse(adata_sub.X):
            adata_sub.X = adata_sub.X.expm1()
        else:
            adata_sub.X = np.expm1(adata_sub.X)

    adata_sub.layers["count"] = adata_sub.X.copy()

    # Spatial coordinates
    if "spatial" in adata_sub.obsm:
        log("Spatial coordinates found")
    elif "X_UMAP" in adata_sub.obsm:
        log("No spatial coordinates, using UMAP as substitute")
        adata_sub.obsm["spatial"] = adata_sub.obsm["X_UMAP"].copy()
    else:
        log("No spatial coordinates or UMAP found", "ERROR")
        return None

    # Annotation
    adata_sub.obs["annotation"] = adata_sub.obs[ANNOTATION_COL].copy()

    # Save
    adata_sub.write_h5ad(output_path)
    log(f"scRNA-seq data prepared: {output_path}")
    log(f"  Cells: {adata_sub.n_obs}, Genes: {adata_sub.n_vars}")

    return output_path


# ========================= Steps 1-5: gsMap Core =========================

def run_gsmap_pipeline(h5ad_path, gwas_path, sample_name, workdir, homolog_file=None):
    """Run gsMap steps 1-5 with checkpoint resumption."""
    os.makedirs(workdir, exist_ok=True)

    # --- Step 1: Find latent representations ---
    step1_done = os.path.exists(os.path.join(workdir, sample_name, "find_latent_representations"))
    if not step1_done:
        try:
            adata_check = sc.read_h5ad(h5ad_path, backed="r")
            if "latent_GVAE" in adata_check.obsm:
                step1_done = True
                log("Step 1 already completed (latent_GVAE found), skipping")
        except Exception:
            pass

    if not step1_done:
        log("=" * 60)
        args = ["run_find_latent_representations",
                "--workdir", workdir,
                "--sample_name", sample_name,
                "--input_hdf5_path", h5ad_path,
                "--annotation", "annotation",
                "--data_layer", "count"]
        if not run_gsmap(args, "Step 1: find_latent_representations"):
            return False
    else:
        log("Step 1 already completed, skipping")

    # --- Step 2: Latent to gene ---
    latent_to_gene_dir = os.path.join(workdir, sample_name, "latent_to_gene")
    if os.path.exists(latent_to_gene_dir) and len(os.listdir(latent_to_gene_dir)) > 0:
        log("Step 2 already completed, skipping")
    else:
        log("=" * 60)
        args = ["run_latent_to_gene",
                "--workdir", workdir,
                "--sample_name", sample_name,
                "--annotation", "annotation",
                "--latent_representation", "latent_GVAE",
                "--num_neighbour", "51",
                "--num_neighbour_spatial", "201"]
        if homolog_file:
            args += ["--homolog_file", homolog_file]
        if not run_gsmap(args, "Step 2: latent_to_gene"):
            return False

    # --- Step 3: Generate LD scores (22 chromosomes in parallel) ---
    log("=" * 60)
    log("Step 3: generate_ldscore (22 chromosomes in parallel)")

    ldscore_dir = os.path.join(workdir, sample_name, "generate_ldscore")
    step3_done = False
    if os.path.exists(ldscore_dir):
        chunks = [f for f in os.listdir(ldscore_dir) if f.startswith(f"{sample_name}_chunk")]
        if chunks:
            sample_chunk = os.path.join(ldscore_dir, chunks[0])
            n_files = len(os.listdir(sample_chunk))
            if n_files >= 66:
                step3_done = True
                log("Step 3 already completed (66 files), skipping")

    if not step3_done:
        from concurrent.futures import ProcessPoolExecutor, as_completed

        chrom_args = [(GSMAP_BIN, workdir, sample_name, c, GSMAP_RESOURCE) for c in range(1, 23)]
        with ProcessPoolExecutor(max_workers=NUM_PARALLEL_CHROMS) as executor:
            futures = {executor.submit(_run_chrom_global, args): args[3] for args in chrom_args}
            for future in as_completed(futures):
                chrom, rc = future.result()
                status = "OK" if rc == 0 else "FAIL"
                log(f"  {status} chr{chrom}")

        # Verify completeness and rerun missing chromosomes
        if os.path.exists(ldscore_dir):
            chunks = [f for f in os.listdir(ldscore_dir) if f.startswith(f"{sample_name}_chunk")]
            if chunks:
                sample_chunk = os.path.join(ldscore_dir, chunks[0])
                n_files = len(os.listdir(sample_chunk))
                if n_files < 66:
                    log(f"Incomplete ({n_files}/66), rerunning missing chromosomes", "WARN")
                    chroms_found = set()
                    for f in os.listdir(sample_chunk):
                        if ".l2.ldscore.feather" in f:
                            chroms_found.add(int(f.split(".")[1]))
                    missing = set(range(1, 23)) - chroms_found
                    if missing:
                        log(f"  Missing chromosomes: {missing}")
                        missing_args = [(GSMAP_BIN, workdir, sample_name, c, GSMAP_RESOURCE) for c in missing]
                        with ProcessPoolExecutor(max_workers=len(missing)) as executor:
                            futures = {executor.submit(_run_chrom_global, args): args[3] for args in missing_args}
                            for future in as_completed(futures):
                                chrom, rc = future.result()
                                status = "OK" if rc == 0 else "FAIL"
                                log(f"  Rerun {status} chr{chrom}")

        log("Step 3 completed")

    # --- Step 4: Spatial LDSC ---
    ldsc_result = os.path.join(workdir, sample_name, "spatial_ldsc",
                               f"{sample_name}_{TRAIT_NAME}.csv.gz")
    if os.path.exists(ldsc_result):
        log("Step 4 already completed, skipping")
    else:
        log("=" * 60)
        args = ["run_spatial_ldsc",
                "--workdir", workdir,
                "--sample_name", sample_name,
                "--trait_name", TRAIT_NAME,
                "--sumstats_file", gwas_path,
                "--w_file", f"{GSMAP_RESOURCE}/LDSC_resource/weights_hm3_no_hla/weights.",
                "--num_processes", str(NUM_PROCESSES_LDSC)]
        if not run_gsmap(args, "Step 4: spatial_ldsc"):
            return False

    # --- Step 5: Cauchy combination ---
    cauchy_result = os.path.join(workdir, sample_name, "cauchy_combination",
                                 f"{sample_name}_{TRAIT_NAME}.Cauchy.csv.gz")
    if os.path.exists(cauchy_result):
        log("Step 5 already completed, skipping")
    else:
        log("=" * 60)
        args = ["run_cauchy_combination",
                "--workdir", workdir,
                "--sample_name", sample_name,
                "--trait_name", TRAIT_NAME,
                "--annotation", "annotation"]
        if not run_gsmap(args, "Step 5: cauchy_combination"):
            return False

    log("gsMap core pipeline completed")
    return True


# ========================= Step 6: Causal Genes =========================

def run_causal_genes(h5ad_path, workdir, sample_name):
    """Compute gene-trait correlations using spatial LDSC p-values."""
    log("=" * 60)
    log("Step 6: Causal gene analysis")
    log("=" * 60)

    output_path = os.path.join(PROJECT_DIR, f"causal_genes_{sample_name}_{TRAIT_NAME}.csv")
    if os.path.exists(output_path):
        log(f"Causal genes already exists, skipping: {output_path}")
        return pd.read_csv(output_path)

    adata = sc.read_h5ad(h5ad_path)
    ldsc_path = os.path.join(workdir, sample_name, "spatial_ldsc",
                             f"{sample_name}_{TRAIT_NAME}.csv.gz")
    spatial_ldsc = pd.read_csv(ldsc_path)

    spatial_ldsc = spatial_ldsc.set_index("spot")
    common = adata.obs_names.intersection(spatial_ldsc.index)
    adata = adata[common, :]
    spatial_ldsc = spatial_ldsc.loc[common]

    trait_score = -np.log10(spatial_ldsc["p"].values + 1e-300)
    X = adata.layers["count"]

    n_genes = adata.n_vars
    correlations = np.zeros(n_genes)
    pvalues = np.ones(n_genes)

    log("Computing gene-trait correlations...")
    batch_size = 1000
    for i in range(0, n_genes, batch_size):
        end = min(i + batch_size, n_genes)
        if hasattr(X, "toarray"):
            batch = X[:, i:end].toarray()
        else:
            batch = X[:, i:end]
        for j in range(batch.shape[1]):
            gene_expr = batch[:, j]
            if gene_expr.std() > 0:
                r, p = stats.pearsonr(gene_expr, trait_score)
                correlations[i + j] = r
                pvalues[i + j] = p
        if (i // batch_size) % 20 == 0:
            log(f"  Progress: {end}/{n_genes}")

    gene_results = pd.DataFrame({
        "gene": adata.var_names,
        "correlation": correlations,
        "pvalue": pvalues,
    })
    gene_results = gene_results[gene_results["correlation"] != 0]
    gene_results = gene_results.sort_values("correlation", ascending=False)

    gene_results.to_csv(output_path, index=False)
    log(f"Causal genes saved: {output_path}")
    log(f"  Top 10: {gene_results.head(10)['gene'].tolist()}")

    return gene_results


# ========================= Step 7: Visualization =========================

def run_visualization(h5ad_path, workdir, sample_name, gene_results=None):
    """Generate publication-quality figures from gsMap results."""
    import matplotlib
    matplotlib.rcParams["font.family"] = "Arial"
    matplotlib.rcParams["pdf.fonttype"] = 42
    import matplotlib.pyplot as plt

    log("=" * 60)
    log("Step 7: Visualization")
    log("=" * 60)

    fig_dir = os.path.join(PROJECT_DIR, "figures", sample_name)
    os.makedirs(fig_dir, exist_ok=True)

    adata = sc.read_h5ad(h5ad_path)
    ldsc_path = os.path.join(workdir, sample_name, "spatial_ldsc",
                             f"{sample_name}_{TRAIT_NAME}.csv.gz")
    cauchy_path = os.path.join(workdir, sample_name, "cauchy_combination",
                               f"{sample_name}_{TRAIT_NAME}.Cauchy.csv.gz")

    spatial_ldsc = pd.read_csv(ldsc_path)
    cauchy = pd.read_csv(cauchy_path)

    spatial_ldsc = spatial_ldsc.set_index("spot")
    common = adata.obs_names.intersection(spatial_ldsc.index)
    adata = adata[common, :]
    spatial_ldsc = spatial_ldsc.loc[common]

    adata.obs = adata.obs.copy()
    adata.obs["gsmap_p"] = spatial_ldsc["p"].values
    adata.obs["gsmap_neglog10p"] = -np.log10(spatial_ldsc["p"].values + 1e-300)
    clip_val = np.percentile(adata.obs["gsmap_neglog10p"], 99)
    adata.obs["gsmap_clip"] = np.clip(adata.obs["gsmap_neglog10p"], 0, clip_val)

    # --- Figure 1: UMAP dual panel ---
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    sc.pl.embedding(adata, basis="X_UMAP", color="annotation",
                    ax=axes[0], show=False, title="Cell Type Annotation",
                    legend_fontsize=6, legend_loc="right margin")
    sc.pl.embedding(adata, basis="X_UMAP", color="gsmap_clip", cmap="Reds",
                    ax=axes[1], show=False,
                    title=f"gsMap: {TRAIT_NAME} Association\n(-log10 P)")
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "Fig1_UMAP_mapping.pdf"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(fig_dir, "Fig1_UMAP_mapping.png"), dpi=300, bbox_inches="tight")
    plt.close()
    log("Figure 1: UMAP mapping saved")

    # --- Figure 2: Cell type association bar chart ---
    fig, ax = plt.subplots(figsize=(10, 8))
    cauchy_sorted = cauchy.sort_values("p_cauchy")
    cauchy_sorted["neglog10p"] = -np.log10(cauchy_sorted["p_cauchy"] + 1e-300)
    n_types = len(cauchy_sorted)
    colors = ["#c0392b" if p < 0.05 / n_types else "#e74c3c" if p < 0.05 else "#bdc3c7"
              for p in cauchy_sorted["p_cauchy"]]
    ax.barh(range(n_types), cauchy_sorted["neglog10p"], color=colors, edgecolor="white")
    ax.set_yticks(range(n_types))
    ax.set_yticklabels(cauchy_sorted["annotation"], fontsize=9)
    ax.set_xlabel("-log10(P value)", fontsize=12)
    ax.set_title(f"Cell Type Association with {TRAIT_NAME}", fontsize=14, fontweight="bold")
    ax.axvline(x=-np.log10(0.05 / n_types), color="red", linestyle="--", linewidth=1, label="Bonferroni")
    ax.axvline(x=-np.log10(0.05), color="orange", linestyle="--", linewidth=1, label="Nominal")
    ax.legend(loc="lower right")
    ax.invert_yaxis()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "Fig2_celltype_association.pdf"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(fig_dir, "Fig2_celltype_association.png"), dpi=300, bbox_inches="tight")
    plt.close()
    log("Figure 2: cell type association bar chart saved")

    # --- Figure 3: Top 5 cell types ---
    top5 = cauchy_sorted.head(5)["annotation"].tolist()
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    for i, ct in enumerate(top5):
        mask = (adata.obs["annotation"] == ct).values
        axes[i].scatter(adata.obsm["X_UMAP"][~mask, 0], adata.obsm["X_UMAP"][~mask, 1],
                        c="lightgrey", s=0.5, alpha=0.3, rasterized=True)
        axes[i].scatter(adata.obsm["X_UMAP"][mask, 0], adata.obsm["X_UMAP"][mask, 1],
                        c=adata.obs.loc[mask, "gsmap_clip"], cmap="Reds", s=2, alpha=0.8, rasterized=True)
        p_val = cauchy_sorted[cauchy_sorted["annotation"] == ct]["p_cauchy"].values[0]
        axes[i].set_title(f"{ct}\nP = {p_val:.2e}", fontsize=9, fontweight="bold")
        axes[i].set_xticks([]); axes[i].set_yticks([])
        for spine in axes[i].spines.values():
            spine.set_visible(False)
    plt.suptitle(f"Top 5 Cell Types Associated with {TRAIT_NAME}", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "Fig3_top5_spatial.pdf"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(fig_dir, "Fig3_top5_spatial.png"), dpi=300, bbox_inches="tight")
    plt.close()
    log("Figure 3: top 5 cell types saved")

    # --- Figure 4: Causal genes bar chart ---
    if gene_results is not None:
        fig, ax = plt.subplots(figsize=(10, 10))
        top30 = gene_results.head(30).copy().iloc[::-1]
        colors = ["#8b0000" if r > 0.5 else "#c0392b" if r > 0.48 else "#e74c3c"
                  for r in top30["correlation"]]
        ax.barh(range(len(top30)), top30["correlation"], color=colors, edgecolor="white")
        ax.set_yticks(range(len(top30)))
        ax.set_yticklabels(top30["gene"], fontsize=10)
        ax.set_xlabel("Pearson Correlation with Trait Association", fontsize=12)
        ax.set_title(f"Top 30 Putative Causal Genes for {TRAIT_NAME}", fontsize=14, fontweight="bold")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, "Fig4_causal_genes.pdf"), dpi=300, bbox_inches="tight")
        plt.savefig(os.path.join(fig_dir, "Fig4_causal_genes.png"), dpi=300, bbox_inches="tight")
        plt.close()
        log("Figure 4: causal genes bar chart saved")

    # --- Figure 5: Volcano plot ---
    if gene_results is not None:
        fig, ax = plt.subplots(figsize=(10, 8))
        genes = gene_results.copy()
        genes["neglog10p"] = np.clip(-np.log10(genes["pvalue"] + 1e-300), 0, 300)

        p_threshold = 0.05 / len(genes)
        corr_threshold = 0.1

        sig_up = (genes["pvalue"] < p_threshold) & (genes["correlation"] > corr_threshold)
        sig_down = (genes["pvalue"] < p_threshold) & (genes["correlation"] < -corr_threshold)
        nonsig = ~sig_up & ~sig_down

        ax.scatter(genes.loc[nonsig, "correlation"], genes.loc[nonsig, "neglog10p"],
                   c="#CCCCCC", s=3, alpha=0.5, rasterized=True, label="Not significant")
        ax.scatter(genes.loc[sig_up, "correlation"], genes.loc[sig_up, "neglog10p"],
                   c="#c0392b", s=5, alpha=0.7, rasterized=True, label=f"Positive (n={sig_up.sum()})")
        ax.scatter(genes.loc[sig_down, "correlation"], genes.loc[sig_down, "neglog10p"],
                   c="#2980b9", s=5, alpha=0.7, rasterized=True, label=f"Negative (n={sig_down.sum()})")

        top_pos = genes[sig_up].nlargest(8, "correlation")
        top_neg = genes[sig_down].nsmallest(8, "correlation")
        for _, row in pd.concat([top_pos, top_neg]).iterrows():
            ax.annotate(row["gene"], (row["correlation"], row["neglog10p"]),
                        fontsize=6, ha="center", va="bottom", fontweight="bold")

        ax.axhline(y=-np.log10(p_threshold), color="red", linestyle="--", linewidth=0.8, alpha=0.5)
        ax.axvline(x=corr_threshold, color="grey", linestyle="--", linewidth=0.8, alpha=0.5)
        ax.axvline(x=-corr_threshold, color="grey", linestyle="--", linewidth=0.8, alpha=0.5)
        ax.set_xlabel("Correlation with Trait Association", fontsize=12)
        ax.set_ylabel("-log10(P value)", fontsize=12)
        ax.set_title(f"Volcano Plot: Causal Genes for {TRAIT_NAME}", fontsize=14, fontweight="bold")
        ax.legend(loc="upper left", fontsize=9)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, "Fig5_volcano.pdf"), dpi=300, bbox_inches="tight")
        plt.savefig(os.path.join(fig_dir, "Fig5_volcano.png"), dpi=300, bbox_inches="tight")
        plt.close()
        log("Figure 5: volcano plot saved")

    # --- Figure 6: Brain region analysis ---
    if REGION_COL and REGION_COL in adata.obs.columns:
        regions = adata.obs[REGION_COL].unique()
        region_results = []
        for region in regions:
            mask = (adata.obs[REGION_COL] == region).values
            pvals = spatial_ldsc.loc[mask, "p"].values
            weights = np.ones(len(pvals)) / len(pvals)
            cauchy_stats = np.sum(weights * np.tan((0.5 - pvals) * np.pi))
            cauchy_p = 0.5 - np.arctan(cauchy_stats) / np.pi
            region_results.append({"Brain Region": region, "p_cauchy": cauchy_p, "n_cells": mask.sum()})

        region_df = pd.DataFrame(region_results).sort_values("p_cauchy")
        region_df["neglog10p"] = -np.log10(region_df["p_cauchy"] + 1e-300)
        region_df.to_csv(os.path.join(fig_dir, "brain_region_results.csv"), index=False)
        log(f"Brain region results:\n{region_df.to_string(index=False)}")

    log(f"\nAll figures saved to: {fig_dir}")


# ========================= Main =========================

def main():
    log("=" * 60)
    log("gsMap Pipeline")
    log(f"Trait: {TRAIT_NAME}")
    log(f"scRNA-seq: {SCRNA_FILE}")
    log(f"Project directory: {PROJECT_DIR}")
    log("=" * 60)

    os.makedirs(PROJECT_DIR, exist_ok=True)

    gwas_path = qc_and_format_gwas()
    if not gwas_path:
        log("GWAS formatting failed", "ERROR")
        return

    h5ad_path = qc_and_format_scrna()
    if not h5ad_path:
        log("scRNA-seq preparation failed", "ERROR")
        return

    sample_name = f"gsmap_{TRAIT_NAME}"
    workdir = os.path.join(PROJECT_DIR, "results")

    homolog_file = None
    if SPECIES == "mouse":
        homolog_file = os.path.join(GSMAP_RESOURCE, "homologs/mouse_human_homologs.txt")
    elif SPECIES == "macaque":
        homolog_file = os.path.join(GSMAP_RESOURCE, "homologs/macaque_human_homologs.txt")

    success = run_gsmap_pipeline(h5ad_path, gwas_path, sample_name, workdir, homolog_file)
    if not success:
        log("gsMap pipeline failed", "ERROR")
        return

    gene_results = run_causal_genes(h5ad_path, workdir, sample_name)
    run_visualization(h5ad_path, workdir, sample_name, gene_results)

    log("=" * 60)
    log("Pipeline completed")
    log(f"Results: {workdir}")
    log(f"Figures: {os.path.join(PROJECT_DIR, 'figures', sample_name)}")
    log("=" * 60)


if __name__ == "__main__":
    main()